# CS781 Phase 2 – Legal NER – Knowledge distillation: teacher Model 4 (GQA+MHA+MoE, LoRMA) → student Model 1 (GQA)

**Attach these inputs before running** (Add Input):
1. The Phase-2 data (`ner_data_clean` from the data-prep notebook).
2. **Student:** the output of your Model-1 Phase-1 notebook (folder `legal-gqa-final`).
3. **Teacher backbone:** the output of your Model-4 Phase-1 notebook (folder `legal-gqamoe-final`).
4. **Teacher adapters:** the output of the Model-4 **LoRMA** NER notebook (folder `ner_model4_gqamoe_lorma` with `phaseA_lorma_weights.pt`, `phaseB_lorma_weights.pt`, `results.json`).

Settings: GPU T4 x2 (one GPU is used), Internet not needed.

**Technique: knowledge distillation.** The teacher (Model 4 + LoRMA, larger and better pre-trained) gives a score for each of the 25 BIO tags at every token (soft labels). The student (Model 1, smaller) is fully fine-tuned with
`loss = (1 - KD_ALPHA) · CE(gold tags) + KD_ALPHA · T² · KL(teacher ‖ student)` (temperature `KD_TEMP`).
* Phase A (student trained on fold1, validated on fold2) uses the teacher trained on fold1 only; Phase B (fold1+fold2 → test) uses the teacher trained on fold1+fold2. So the fold2 score stays honest.
* `KD_ALPHA = 0` switches distillation off and trains the same student with gold labels only: run it once as the **no-KD baseline** for the report.
* `KD_ON_TEST_TEXT = True` also distils on the unlabelled test documents (teacher predictions only, no labels). Off by default; check that your course rules allow it.
* `FT_MODE = "full"` fine-tunes all student weights; `"partial"` trains only the last `N_TRAIN_LAYERS` blocks.

**Real run:** set `SMOKE_TEST = False`, then Save & Run All. A session time cap (`MAX_SESSION_HOURS`) stops training cleanly and saves everything; to continue, add this notebook's own output as an input and run again (it resumes automatically).

Flow: **Phase A** train on fold1, validate on fold2 (pick best epoch, error analysis) → **Phase B** refit on fold1+fold2 for that many epochs → predict fold3 → `submission.json`.

In [ ]:
import os, sys, glob, json, time, math, random, shutil
SESSION_START = time.time()
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ======================= SETTINGS (change here) =======================
EXPERIMENT_NAME = "kd_model4_lorma_to_model1"

# --- windows ---
MAX_LEN = 1024           # tokens per training / inference window (the Phase-1 model was trained on 1024)
OVERLAP = 256            # tokens re-read at the start of the next window (warm-up context, causal model)
HEADER_TOKENS = 256      # first N tokens of the document (case header: who is appellant / respondent) are put in
                         # front of every later window as context. 0 = off. Needs HEADER_TOKENS + OVERLAP < MAX_LEN.
ADD_EOS_PREFIX = True    # put the <eos> token in front of every document (that is how Phase-1 separated documents)

# --- training ---
BATCH_SIZE = 4            # windows per optimizer step
MICRO_BATCH = 4          # windows per forward pass (lower it, e.g. to 2 or 1, if the GPU runs out of memory; steps are accumulated)
EPOCHS = 10
LR_BODY = 1e-4           # learning rate of the Phase-1 backbone weights
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
PATIENCE = 3             # stop if validation F1 does not improve for this many epochs
NEG_KEEP_PROB = 1.0      # chance of keeping a training window that has NO entity (1.0 = keep all)
CLASS_WEIGHTS = "sqrt"   # "none" | "sqrt" | "inv"

# --- what is fine-tuned (the student, Model 1) ---
FT_MODE = "full"         # "full" = all backbone weights | "partial" = only the last N_TRAIN_LAYERS blocks (+ final norm + head)
N_TRAIN_LAYERS = 4       # used when FT_MODE = "partial"
FREEZE_EMBEDDINGS = False  # True = keep the token embeddings fixed (always fixed in "partial" mode)
USE_CHECKPOINT = False   # activation checkpointing: ~30% slower, much less GPU memory (turn on if the smoke test runs out of memory)

# --- knowledge distillation ---
KD_ALPHA = 0.5           # weight of the teacher term (0 = no distillation, gold labels only = baseline)
KD_TEMP = 2.0            # softmax temperature for teacher and student scores
KD_ON_TEST_TEXT = False  # also distil on the unlabelled fold3 (test) text. Off by default.

# --- head / decoding ---
HEAD = "bilstm"          # "linear" or "bilstm" (bilstm gives the causal model some right-side context)
DECODE = "viterbi"       # "viterbi" or "argmax"
VOTE = "auto"            # same-name voting inside a document for APP/RESP and A.COUNSEL/R.COUNSEL.
                         # "auto" = use it only if it improves the fold2 score, True = always, False = never

# --- smoke test ---
SMOKE_TEST = False       # True = quick end-to-end check (few docs, 2 epochs, minutes). False = the real full run.
SMOKE_DOCS = 3           # documents used per fold in smoke mode
SMOKE_MAX_WINDOWS = 24   # training windows per epoch in smoke mode
SMOKE_EPOCHS = 2

# --- session time cap and resume (for "Save & Run All" commits) ---
MAX_SESSION_HOURS = 8.0    # stop training cleanly after this long (Kaggle commit limit is 12 h); progress is saved
RESERVE_MIN = 20           # minutes kept free for the final steps (test prediction, saving)
RESUME_FROM_INPUT = True   # continue from the output of an earlier session if it is attached as an input

# --- protocol ---
REFIT_ON_ALL = True      # Phase B: retrain on fold1+fold2 for the best epoch count, then predict the test set
SEED = 42

# --- paths (None = search /kaggle/input automatically) ---
DATA_DIR = None
MODEL_DIR = None
TOKENIZER_DIR = None
TEACHER_BASE_DIR = None      # Model-4 Phase-1 folder (legal-gqamoe-final)
TEACHER_ADAPTER_DIR = None   # Model-4 LoRMA NER output folder
OUT_DIR = "/kaggle/working/ner_kd_model4_to_model1"
# [[END OF SETTINGS]]

for name in ["DATA_DIR", "MODEL_DIR", "TOKENIZER_DIR", "OUT_DIR", "TEACHER_BASE_DIR", "TEACHER_ADAPTER_DIR"]:
    env_value = os.environ.get("NER_" + name)
    if env_value:
        globals()[name] = env_value
OUT_OF_TIME = False
USE_VOTE = False
FULL_EPOCHS = EPOCHS
assert FT_MODE in ("full", "partial"), "FT_MODE must be full or partial"
assert 0.0 <= KD_ALPHA <= 1.0, "KD_ALPHA must be between 0 and 1"
assert HEADER_TOKENS == 0 or HEADER_TOKENS + OVERLAP < MAX_LEN, "HEADER_TOKENS + OVERLAP must be smaller than MAX_LEN"
if SMOKE_TEST:
    EPOCHS = SMOKE_EPOCHS
    OUT_DIR = OUT_DIR.rstrip("/") + "_smoke"          # never overwrite real outputs
    print("=" * 60)
    print("SMOKE TEST MODE: tiny subset, results are NOT meaningful. Set SMOKE_TEST = False for the real run.")
    print("=" * 60)
if SMOKE_TEST and os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)                           # a smoke test always starts clean
os.makedirs(OUT_DIR, exist_ok=True)

def settings_fingerprint():
    # everything that changes the result of training; a resume is only allowed when this matches
    names = ["MAX_LEN", "OVERLAP", "ADD_EOS_PREFIX", "BATCH_SIZE", "EPOCHS", "LR_BODY", "LR_HEAD", "WEIGHT_DECAY",
             "WARMUP_FRAC", "PATIENCE", "NEG_KEEP_PROB", "CLASS_WEIGHTS", "FT_MODE", "N_TRAIN_LAYERS",
             "FREEZE_EMBEDDINGS", "KD_ALPHA", "KD_TEMP", "KD_ON_TEST_TEXT", "HEAD", "DECODE", "REFIT_ON_ALL",
             "SEED", "SMOKE_TEST", "HEADER_TOKENS"]
    fp = {}
    for n in names:
        fp[n] = globals()[n]
    return json.loads(json.dumps(fp))

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("device:", DEVICE, "| torch", torch.__version__)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

# hide a harmless transformers warning about the custom model class (the weights are verified when loading)
import logging
from transformers.utils import logging as hf_logging
class _DropClassWarning(logging.Filter):
    def filter(self, record):
        return "to instantiate a model of type" not in record.getMessage()
for _h in hf_logging._get_library_root_logger().handlers:
    _h.addFilter(_DropClassWarning())

## 0b. Resume from an earlier session (automatic)

In [ ]:
def find_resume_source():
    if SMOKE_TEST or not RESUME_FROM_INPUT:
        return None
    if os.path.exists(os.path.join(OUT_DIR, "resume_marker.json")):
        print("resume: local progress found in", OUT_DIR, "-> continuing from it")
        return None
    candidates = []
    for path in glob.glob("/kaggle/input/**/resume_marker.json", recursive=True):
        if "_smoke" in path:
            continue
        try:
            marker = json.load(open(path))
        except Exception:
            continue
        candidates.append((marker.get("saved_at", 0), path, marker))
    if not candidates:
        print("resume: nothing to resume from -> fresh start")
        return None
    candidates.sort(key=lambda x: x[0], reverse=True)
    saved_at, path, marker = candidates[0]
    if marker.get("status") == "complete":
        print("resume: the attached output is from a COMPLETED run -> fresh start (", path, ")")
        return None
    if marker.get("fingerprint") != settings_fingerprint():
        old = marker.get("fingerprint", {})
        new = settings_fingerprint()
        diff = {k: (old.get(k), new.get(k)) for k in new if old.get(k) != new.get(k)}
        print("resume: attached output was made with DIFFERENT settings -> NOT resuming. Differences (old, new):", diff)
        return None
    return os.path.dirname(path), marker

source = find_resume_source()
if source is not None:
    folder, marker = source
    print("resume: copying progress from", folder, "| last status:", marker.get("status"), "|", marker.get("note"), "|", marker.get("saved_at_text"))
    for fname in os.listdir(folder):
        if fname.endswith(".tmp"):
            continue
        src_path = os.path.join(folder, fname)
        if os.path.isfile(src_path):
            shutil.copy2(src_path, os.path.join(OUT_DIR, fname))
    print("resume: restored files:", sorted(os.listdir(OUT_DIR)))

## 1. Model code (copied unchanged from the Phase-1 notebooks: Model 1 (student) and Model 4 (teacher) share these three files)

In [ ]:
%%writefile /kaggle/working/legal_gqa_model.py
"""
Model 1 for CS781 Phase-1: GQA + RoPE + SwiGLU + RMSNorm decoder-only transformer.

Wrapped as a transformers PretrainedConfig/PreTrainedModel subclass so it is a
drop-in replacement for GPT2Config/GPT2LMHeadModel in the starter notebook:
same save_pretrained/from_pretrained, same Trainer + TrainingArguments +
resume_from_checkpoint behavior, same model.generate() call signature.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast


class LegalGQAConfig(PretrainedConfig):
    model_type = "legal_gqa"

    def __init__(
        self,
        vocab_size=50257,
        n_positions=1024,
        n_embd=1024,
        n_layer=8,
        n_head=16,
        n_kv_head=4,
        d_ff=2688,
        rope_theta=10000.0,
        dropout=0.1,
        initializer_range=0.02,
        rms_norm_eps=1e-5,
        bos_token_id=50256,
        eos_token_id=50256,
        pad_token_id=50256,
        activation_checkpointing=False,
        tie_word_embeddings=True,
        **kwargs,
    ):
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.vocab_size = vocab_size
        self.n_positions = n_positions
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.d_ff = d_ff
        self.rope_theta = rope_theta
        self.dropout = dropout
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.use_cache = True
        # own implementation (HF's gradient_checkpointing flag is not supported here)
        self.activation_checkpointing = activation_checkpointing
        super().__init__(
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings,
            **kwargs,
        )


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k, cos, sin):
    # q, k: (B, n_head, T, head_dim); cos/sin: (T, head_dim)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q_rot = (q * cos) + (rotate_half(q) * sin)
    k_rot = (k * cos) + (rotate_half(k) * sin)
    return q_rot, k_rot


class GroupedQueryAttention(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.head_dim = config.n_embd // config.n_head
        self.n_rep = self.n_head // self.n_kv_head
        self.dropout_p = config.dropout

        self.q_proj = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.n_head * self.head_dim, config.n_embd, bias=False)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        q, k = apply_rope(q, k, cos, sin)
        # RoPE tables are fp32, v comes out of a linear (fp16 under autocast): force one dtype
        q, k = q.to(v.dtype), k.to(v.dtype)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        present = (k, v) if use_cache else None

        # GQA: broadcast each KV head across its group of query heads
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_bias,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=is_causal,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)
        return self.o_proj(out), present


class SwiGLU(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.gate_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.up_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.down_proj = nn.Linear(config.d_ff, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class DecoderBlock(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.attn = GroupedQueryAttention(config)
        self.ln_2 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.ffn = SwiGLU(config)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        attn_out, present = self.attn(
            self.ln_1(x), cos, sin, attn_bias, is_causal, past_key_value, use_cache
        )
        x = x + attn_out
        x = x + self.ffn(self.ln_2(x))
        return x, present


class LegalGQAForCausalLM(PreTrainedModel):
    config_class = LegalGQAConfig
    # input embedding and output head share one weight; must be declared or
    # save_pretrained / Trainer checkpointing refuses to save
    _tied_weights_keys = {"lm_head.weight": "wte.weight"}

    def __init__(self, config: LegalGQAConfig):
        super().__init__(config)
        self.head_dim = config.n_embd // config.n_head

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # weight tying

        self.post_init()

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        # scaled init on residual-stream output projections (GPT-2 recipe)
        for name, p in module.named_parameters(recurse=False):
            if name in ("o_proj.weight", "down_proj.weight"):
                nn.init.normal_(
                    p, mean=0.0, std=std / math.sqrt(2 * self.config.n_layer)
                )

    def get_input_embeddings(self):
        return self.wte

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def set_input_embeddings(self, value):
        self.wte = value

    def _rope_slice(self, past_len, T, device, dtype):
        # Computed on the fly (float32, then cast). Deliberately NOT stored as a buffer:
        # non-persistent buffers come back uninitialized after from_pretrained().
        inv_freq = 1.0 / (
            self.config.rope_theta
            ** (torch.arange(0, self.head_dim, 2, device=device).float() / self.head_dim)
        )
        pos = torch.arange(past_len, past_len + T, device=device).float()
        emb = torch.cat((torch.outer(pos, inv_freq),) * 2, dim=-1)
        return emb.cos().to(dtype), emb.sin().to(dtype)

    def _build_attn_bias(self, attention_mask, past_len, T, device, dtype):
        """Returns (attn_bias, is_causal). attn_bias is None when the fast
        SDPA is_causal fused-kernel path is usable (no padding to mask)."""
        no_padding = attention_mask is None or bool(attention_mask.bool().all())
        if no_padding:
            return None, T > 1

        total_len = past_len + T
        pad = attention_mask[:, None, None, :].to(dtype)
        bias = (1.0 - pad) * torch.finfo(dtype).min

        if T > 1:
            q_pos = torch.arange(past_len, past_len + T, device=device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=device).unsqueeze(0)
            causal = (k_pos > q_pos).to(dtype) * torch.finfo(dtype).min
            bias = bias + causal[None, None, :, :]
        return bias, False

    def forward(
        self,
        input_ids,
        attention_mask=None,
        past_key_values=None,
        labels=None,
        use_cache=None,
    ):
        # NOTE: no **kwargs here on purpose. If forward() accepts **kwargs, Trainer assumes the
        # model normalises its own loss, then multiplies it by n_gpu and skips the
        # gradient-accumulation division (loss reported 2x-64x too high).
        #
        # fp16 autocast is switched on HERE rather than relying on Trainer: under nn.DataParallel
        # on Kaggle the matmuls were measurably running in fp32 (no tensor cores, ~4x slower).
        # Only used for unpadded batches (all of training/eval); padded generation keeps the
        # exact fp32 path, since fp16 additive masks can overflow to -inf.
        amp = input_ids.is_cuda and (attention_mask is None or bool(attention_mask.bool().all()))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            return self._forward_impl(input_ids, attention_mask, past_key_values, labels, use_cache)

    def _forward_impl(self, input_ids, attention_mask, past_key_values, labels, use_cache):
        use_cache = self.config.use_cache if use_cache is None else use_cache
        if self.training:
            use_cache = False  # never keep K/V during training
        B, T = input_ids.shape
        past_len = past_key_values[0][0].shape[2] if past_key_values is not None else 0

        x = self.drop(self.wte(input_ids))
        cos, sin = self._rope_slice(past_len, T, input_ids.device, x.dtype)
        attn_bias, is_causal = self._build_attn_bias(
            attention_mask, past_len, T, input_ids.device, x.dtype
        )

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            pkv = past_key_values[i] if past_key_values is not None else None
            if self.training and self.config.activation_checkpointing:
                x = torch_checkpoint(
                    lambda h, blk=block: blk(h, cos, sin, attn_bias, is_causal, None, False)[0],
                    x,
                    use_reentrant=False,
                )
            else:
                x, present = block(x, cos, sin, attn_bias, is_causal, pkv, use_cache)
                if use_cache:
                    presents.append(present)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        # when labels are given only the loss is needed; returning the full
        # (B, T, vocab) logits would just be gathered across GPUs for nothing
        return CausalLMOutputWithPast(
            loss=loss,
            logits=None if labels is not None else logits,
            past_key_values=tuple(presents) if use_cache else None,
        )

    # No prepare_inputs_for_generation / GenerationMixin here on purpose —
    # generation is done by generate_utils.generate_batch(), which calls
    # this model's forward() directly with its own tuple-based KV cache
    # rather than going through model.generate().

In [ ]:
%%writefile /kaggle/working/moe.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class Expert(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class MoE(nn.Module):
    def __init__(self, d_model, n_routed, n_shared, top_k, expert_d_ff, aux_loss_coef=0.01):
        super().__init__()
        self.n_routed, self.top_k = n_routed, top_k
        self.aux_loss_coef = aux_loss_coef
        self.router = nn.Linear(d_model, n_routed, bias=False)
        self.routed_experts = nn.ModuleList([Expert(d_model, expert_d_ff) for _ in range(n_routed)])
        self.shared_experts = nn.ModuleList([Expert(d_model, expert_d_ff) for _ in range(n_shared)])
        self.last_aux_loss = None

    def forward(self, x):
        B, T, D = x.shape
        flat = x.reshape(-1, D)                                 # (N, D), N = B*T
        N = flat.shape[0]

        logits = self.router(flat)                               # (N, n_routed)
        probs = F.softmax(logits, dim=-1)
        top_w, top_i = probs.topk(self.top_k, dim=-1)             # (N, top_k) each
        top_w = top_w / top_w.sum(dim=-1, keepdim=True)           # renormalize among selected

        out = torch.zeros_like(flat)
        for e, expert in enumerate(self.routed_experts):
            token_slot, k_slot = (top_i == e).nonzero(as_tuple=True)   # which tokens picked expert e, and at which of their top_k slots
            if token_slot.numel() == 0:
                continue
            expert_out = expert(flat[token_slot])                       # only THIS expert's assigned tokens, real compute saving
            out.index_add_(0, token_slot, expert_out * top_w[token_slot, k_slot].unsqueeze(-1))

        for expert in self.shared_experts:
            out = out + expert(flat)                                    # always on, every token

        if self.training:
            # Switch-Transformer style load-balancing aux loss: encourages the router
            # to spread tokens evenly across experts rather than collapsing onto a few.
            with torch.no_grad():
                assign_count = torch.zeros(self.n_routed, device=x.device)
                assign_count.scatter_add_(0, top_i.reshape(-1), torch.ones_like(top_i.reshape(-1), dtype=torch.float))
                frac_tokens = assign_count / (N * self.top_k)
            frac_prob = probs.mean(dim=0)
            self.last_aux_loss = self.aux_loss_coef * self.n_routed * (frac_tokens * frac_prob).sum()
        else:
            self.last_aux_loss = None

        return out.view(B, T, D)

In [ ]:
%%writefile /kaggle/working/model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

from legal_gqa_model import RMSNorm, GroupedQueryAttention, LegalGQAConfig
from moe import MoE


class LegalGQAMoEConfig(PretrainedConfig):
    model_type = "legal_gqa_mha_moe"

    def __init__(
        self,
        vocab_size=50257, n_positions=1024, n_embd=1024, n_layer=9,
        mha_layers=(0, 4, 8),           # 0-indexed: layers 1, 5, 9 -- full MHA (n_kv_head = n_head)
        n_head=16, gqa_n_kv_head=4,     # the other 6 layers: GQA, same n_kv_head model 1/3 already proved
        n_routed_experts=6, n_shared_experts=1, moe_top_k=2, expert_d_ff=896, moe_aux_loss_coef=0.01,
        rope_theta=10000.0,
        dropout=0.0, initializer_range=0.02, rms_norm_eps=1e-5,
        bos_token_id=50256, eos_token_id=50256, pad_token_id=50256,
        activation_checkpointing=False, tie_word_embeddings=True,
        **kwargs,
    ):
        assert n_embd % n_head == 0
        self.vocab_size, self.n_positions, self.n_embd, self.n_layer = vocab_size, n_positions, n_embd, n_layer
        self.mha_layers = list(mha_layers)
        self.n_head, self.gqa_n_kv_head = n_head, gqa_n_kv_head
        self.n_routed_experts, self.n_shared_experts = n_routed_experts, n_shared_experts
        self.moe_top_k, self.expert_d_ff, self.moe_aux_loss_coef = moe_top_k, expert_d_ff, moe_aux_loss_coef
        self.rope_theta = rope_theta
        self.dropout, self.initializer_range, self.rms_norm_eps = dropout, initializer_range, rms_norm_eps
        self.use_cache = True
        self.activation_checkpointing = activation_checkpointing
        super().__init__(
            bos_token_id=bos_token_id, eos_token_id=eos_token_id, pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings, **kwargs,
        )


class MixedDecoderBlock(nn.Module):
    def __init__(self, config: LegalGQAMoEConfig, layer_idx: int):
        super().__init__()
        self.layer_idx = layer_idx
        self.is_mha = layer_idx in config.mha_layers
        n_kv_head = config.n_head if self.is_mha else config.gqa_n_kv_head
        # GroupedQueryAttention reads n_head/n_kv_head/n_embd/dropout off a config
        # object -- reusing it unmodified means building one small per-layer config
        # with just n_kv_head varied, rather than touching model 1's tested file at all
        attn_config = LegalGQAConfig(
            vocab_size=config.vocab_size, n_positions=config.n_positions,
            n_embd=config.n_embd, n_head=config.n_head, n_kv_head=n_kv_head,
            dropout=config.dropout,
        )
        self.ln_1 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.attn = GroupedQueryAttention(attn_config)
        self.ln_2 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.moe = MoE(config.n_embd, config.n_routed_experts, config.n_shared_experts,
                       config.moe_top_k, config.expert_d_ff, config.moe_aux_loss_coef)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_kv=None, use_cache=False):
        attn_out, present = self.attn(self.ln_1(x), cos, sin, attn_bias, is_causal, past_kv, use_cache)
        x = x + attn_out
        moe_out = self.moe(self.ln_2(x))
        x = x + moe_out
        return x, present, self.moe.last_aux_loss


class LegalGQAMoEForCausalLM(PreTrainedModel):
    config_class = LegalGQAMoEConfig
    _tied_weights_keys = {"lm_head.weight": "wte.weight"}

    def __init__(self, config: LegalGQAMoEConfig):
        super().__init__(config)
        self.head_dim = config.n_embd // config.n_head
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([MixedDecoderBlock(config, i) for i in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        self.post_init()

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        for name, p in module.named_parameters(recurse=False):
            if name in ("o_proj.weight", "down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * self.config.n_layer))

    def get_input_embeddings(self):
        return self.wte

    def set_input_embeddings(self, value):
        self.wte = value

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def _rope_slice(self, past_len, T, device, dtype):
        inv_freq = 1.0 / (self.config.rope_theta ** (torch.arange(0, self.head_dim, 2, device=device).float() / self.head_dim))
        pos = torch.arange(past_len, past_len + T, device=device).float()
        emb = torch.cat((torch.outer(pos, inv_freq),) * 2, dim=-1)
        return emb.cos().to(dtype), emb.sin().to(dtype)

    def _build_attn_bias(self, attention_mask, past_len, T, device, dtype):
        no_padding = attention_mask is None or bool(attention_mask.bool().all())
        if no_padding:
            return None, T > 1
        total_len = past_len + T
        pad = attention_mask[:, None, None, :].to(dtype)
        bias = (1.0 - pad) * torch.finfo(dtype).min
        if T > 1:
            q_pos = torch.arange(past_len, past_len + T, device=device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=device).unsqueeze(0)
            causal = (k_pos > q_pos).to(dtype) * torch.finfo(dtype).min
            bias = bias + causal[None, None]
        return bias, False

    def forward(self, input_ids, attention_mask=None, past_key_values=None, labels=None, use_cache=None):
        amp = input_ids.is_cuda and (attention_mask is None or bool(attention_mask.bool().all()))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            return self._forward_impl(input_ids, attention_mask, past_key_values, labels, use_cache)
        # no **kwargs above on purpose -- Trainer treats a forward() accepting **kwargs
        # as self-normalizing its loss, breaking grad-accum scaling (models 1/2/3's note)

    def _forward_impl(self, input_ids, attention_mask, past_key_values, labels, use_cache):
        use_cache = self.config.use_cache if use_cache is None else use_cache
        if self.training:
            use_cache = False
        B, T = input_ids.shape
        past_len = past_key_values[0][0].shape[2] if past_key_values is not None else 0

        x = self.drop(self.wte(input_ids))
        cos, sin = self._rope_slice(past_len, T, input_ids.device, x.dtype)
        attn_bias, is_causal = self._build_attn_bias(attention_mask, past_len, T, input_ids.device, x.dtype)

        presents = [] if use_cache else None
        moe_losses = []
        for block in self.blocks:
            pkv = past_key_values[block.layer_idx] if past_key_values is not None else None
            if self.training and self.config.activation_checkpointing:
                def run(h, blk=block):
                    h2, _, moe_l = blk(h, cos, sin, attn_bias, is_causal, None, False)
                    if moe_l is not None:
                        moe_losses.append(moe_l)
                    return h2
                x = torch_checkpoint(run, x, use_reentrant=False)
            else:
                x, present, moe_l = block(x, cos, sin, attn_bias, is_causal, pkv, use_cache)
                if use_cache:
                    presents.append(present)
                if moe_l is not None:
                    moe_losses.append(moe_l)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100,
            )
            if moe_losses:
                loss = loss + torch.stack(moe_losses).mean()

        return CausalLMOutputWithPast(
            loss=loss,
            logits=None if labels is not None else logits,
            past_key_values=tuple(presents) if use_cache else None,
        )

## 2. Find data, model and tokenizer

In [ ]:
def find_files(filename):
    return sorted(glob.glob("/kaggle/input/**/" + filename, recursive=True))

def pick_data_file(filename):
    if DATA_DIR is not None:
        found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    else:
        found = find_files(filename)
    if not found:
        raise FileNotFoundError(filename + " not found. Attach the competition data as an input.")
    # prefer the cleaned copy (output of the data-prep notebook) over any raw copy
    found = sorted(found, key=lambda p: ("ner_data_clean" not in p, p))
    return found[0]

FOLD1_TEXT = pick_data_file("fold1-text.json")
FOLD1_GOLD = pick_data_file("fold1-gold.json")
FOLD2_TEXT = pick_data_file("fold2-text.json")
FOLD2_GOLD = pick_data_file("fold2-gold.json")
FOLD3_TEXT = pick_data_file("fold3-text.json")
LABELS_TXT = pick_data_file("labels.txt")
print("fold1:", FOLD1_TEXT)
print("fold3:", FOLD3_TEXT)

def find_model_dir():
    if MODEL_DIR is not None:
        return MODEL_DIR
    found = []
    for cfg_path in find_files("config.json"):
        folder = os.path.dirname(cfg_path)
        try:
            model_type = json.load(open(cfg_path)).get("model_type")
        except Exception:
            continue
        has_weights = os.path.exists(os.path.join(folder, "model.safetensors")) or \
                      os.path.exists(os.path.join(folder, "pytorch_model.bin"))
        if model_type == "legal_gqa" and has_weights:
            found.append(folder)
    if not found:
        raise FileNotFoundError("No Phase-1 model folder found. Attach the output of the Model-1 notebook (the student).")
    final_ones = [d for d in found if "checkpoint-" not in d]
    if final_ones:
        return final_ones[0]
    # only checkpoints available: take the newest one
    return max(found, key=lambda d: int(d.rsplit("-", 1)[1]))

MODEL_PATH = find_model_dir()
print("Phase-1 model:", MODEL_PATH)

def find_tokenizer_dir():
    if TOKENIZER_DIR is not None:
        return TOKENIZER_DIR
    if os.path.exists(os.path.join(MODEL_PATH, "tokenizer.json")):
        return MODEL_PATH
    found = [os.path.dirname(p) for p in find_files("tokenizer.json") if "checkpoint-" not in p]
    if not found:
        raise FileNotFoundError("No tokenizer.json found. Attach your tokenizer dataset.")
    return found[0]

TOK_PATH = find_tokenizer_dir()
print("Tokenizer:", TOK_PATH)

## 3. Load data (and detect whether the gold end offset is exclusive or inclusive)

In [ ]:
def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def load_texts(path):
    raw = read_json(path)
    texts = {}
    if isinstance(raw, dict):
        for key, value in raw.items():
            if isinstance(value, dict):
                value = value.get("text", "")
            texts[str(key)] = value
    else:
        for item in raw:
            key = item.get("id", item.get("doc_id"))
            texts[str(key)] = item["text"]
    return texts

def first_key(d, names):
    for n in names:
        if n in d:
            return d[n]
    raise KeyError("none of " + str(names) + " in " + str(list(d.keys())))

def parse_annotation(a):
    # returns (start, end, label, text_or_None) exactly as stored in the file
    if isinstance(a, dict):
        s = first_key(a, ["start", "start_char", "start_offset", "begin"])
        e = first_key(a, ["end", "end_char", "end_offset"])
        lab = first_key(a, ["label", "type", "entity_label"])
        txt = a.get("text", a.get("entity_text", None))
        return int(s), int(e), lab, txt
    txt = a[3] if len(a) > 3 else None
    return int(a[0]), int(a[1]), a[2], txt

def load_gold_raw(path):
    raw = read_json(path)
    gold = {}
    if isinstance(raw, dict):
        items = raw.items()
    else:
        items = [(it.get("id", it.get("doc_id")), it) for it in raw]
    for key, value in items:
        if isinstance(value, dict):
            value = value.get("entities", value.get("annotations", value.get("spans", [])))
        gold[str(key)] = [parse_annotation(a) for a in value]
    return gold

texts1 = load_texts(FOLD1_TEXT)
texts2 = load_texts(FOLD2_TEXT)
texts3 = load_texts(FOLD3_TEXT)
gold1_raw = load_gold_raw(FOLD1_GOLD)
gold2_raw = load_gold_raw(FOLD2_GOLD)
print("docs: fold1", len(texts1), "| fold2", len(texts2), "| fold3 (test)", len(texts3))

# ---- labels ----
DEFAULT_LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
LABELS = []
for line in open(LABELS_TXT, encoding="utf-8").read().splitlines():
    line = line.strip()
    if line:
        LABELS.append(line.replace(",", " ").split()[0])
if len(LABELS) != 12 or set(LABELS) != set(DEFAULT_LABELS):
    print("WARNING: labels.txt gave", LABELS, "-> using the 12 official labels instead")
    LABELS = DEFAULT_LABELS
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
N_TAGS = 1 + 2 * len(LABELS)      # O, then B-x / I-x for each label
TAG_NAMES = ["O"]
for l in LABELS:
    TAG_NAMES.append("B-" + l)
    TAG_NAMES.append("I-" + l)
print("labels:", LABELS)

# ---- end offset convention ----
exclusive_hits = 0
inclusive_hits = 0
with_text = 0
for texts, gold in [(texts1, gold1_raw), (texts2, gold2_raw)]:
    for doc_id, anns in gold.items():
        if doc_id not in texts:
            continue
        for s, e, lab, ent_text in anns:
            if ent_text is None:
                continue
            with_text += 1
            if texts[doc_id][s:e] == ent_text:
                exclusive_hits += 1
            if texts[doc_id][s:e + 1] == ent_text:
                inclusive_hits += 1
print(f"entity text check: {with_text} annotations | text[s:e]==ent: {exclusive_hits} | text[s:e+1]==ent: {inclusive_hits}")
if with_text == 0:
    print("WARNING: gold has no entity text, assuming END IS EXCLUSIVE (python slicing)")
    END_INCLUSIVE = False
else:
    END_INCLUSIVE = inclusive_hits > exclusive_hits
print("gold end offset is", "INCLUSIVE" if END_INCLUSIVE else "EXCLUSIVE")

def normalise_gold(gold_raw):
    # internal format everywhere: (start, end_exclusive, label)
    out = {}
    for doc_id, anns in gold_raw.items():
        spans = []
        for s, e, lab, ent_text in anns:
            if lab not in LABEL2ID:
                continue
            spans.append((s, e + 1 if END_INCLUSIVE else e, lab))
        out[doc_id] = spans
    return out

gold1 = normalise_gold(gold1_raw)
gold2 = normalise_gold(gold2_raw)

label_counts = {l: 0 for l in LABELS}
for spans in list(gold1.values()) + list(gold2.values()):
    for s, e, lab in spans:
        label_counts[lab] += 1
print("entity counts (fold1+fold2):", label_counts)

## 4. Tokenise, align characters to tokens, BIO tags

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(TOK_PATH)
tokenizer.model_max_length = 10**9          # we cut windows ourselves
assert tokenizer.is_fast, "need a fast tokenizer for offset mapping"
print("vocab:", len(tokenizer), "| eos:", tokenizer.eos_token_id, "| pad:", tokenizer.pad_token_id)

EOS_ID = tokenizer.eos_token_id
if tokenizer.pad_token_id is not None:
    PAD_ID = tokenizer.pad_token_id
elif EOS_ID is not None:
    PAD_ID = EOS_ID
else:
    PAD_ID = 0

def encode_doc(text):
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = list(enc["input_ids"])
    starts = []
    ends = []
    real = []
    for s, e in enc["offset_mapping"]:
        # byte-level BPE offsets can contain the leading space: trim whitespace
        while s < e and text[s].isspace():
            s += 1
        while e > s and text[e - 1].isspace():
            e -= 1
        starts.append(s)
        ends.append(e)
        real.append(e > s)
    if ADD_EOS_PREFIX and EOS_ID is not None:
        ids = [EOS_ID] + ids
        starts = [0] + starts
        ends = [0] + ends
        real = [False] + real
    doc = {}
    doc["ids"] = np.array(ids, dtype=np.int64)
    doc["starts"] = np.array(starts, dtype=np.int64)
    doc["ends"] = np.array(ends, dtype=np.int64)
    doc["real"] = np.array(real, dtype=bool)
    return doc

def make_tags(doc, spans):
    # spans: (start, end_exclusive, label). Longest span wins if two overlap.
    n = len(doc["ids"])
    tags = np.zeros(n, dtype=np.int64)
    taken = np.zeros(n, dtype=bool)
    ordered = sorted(spans, key=lambda x: -(x[1] - x[0]))
    for s, e, lab in ordered:
        first = int(np.searchsorted(doc["ends"], s, side="right"))   # first token ending after s
        last = int(np.searchsorted(doc["starts"], e, side="left"))    # tokens starting before e are < last
        idx = [i for i in range(first, last) if doc["real"][i] and not taken[i]]
        if not idx:
            continue
        li = LABEL2ID[lab]
        for k, i in enumerate(idx):
            tags[i] = 1 + 2 * li if k == 0 else 2 + 2 * li
            taken[i] = True
    tags[~doc["real"]] = -100
    return tags

def tags_to_spans(tags, starts, ends):
    # tags / starts / ends only for REAL tokens. An I-x without matching B-x/I-x before it starts a new span.
    spans = []
    cur = None
    cs = 0
    ce = 0
    for i in range(len(tags)):
        t = int(tags[i])
        if t == 0:
            if cur is not None:
                spans.append((int(cs), int(ce), LABELS[cur]))
                cur = None
            continue
        lab = (t - 1) // 2
        is_begin = (t - 1) % 2 == 0
        if is_begin or cur != lab:
            if cur is not None:
                spans.append((int(cs), int(ce), LABELS[cur]))
            cur = lab
            cs = starts[i]
        ce = ends[i]
    if cur is not None:
        spans.append((int(cs), int(ce), LABELS[cur]))
    return spans

def build_docs(texts, gold=None):
    docs = []
    for doc_id, text in texts.items():
        doc = encode_doc(text)
        doc["id"] = doc_id
        doc["n_chars"] = len(text)
        doc["text"] = text
        if gold is not None:
            doc["gold"] = gold.get(doc_id, [])
            doc["tags"] = make_tags(doc, doc["gold"])
        docs.append(doc)
    return docs

t0 = time.time()
docs1 = build_docs(texts1, gold1)
docs2 = build_docs(texts2, gold2)
docs3 = build_docs(texts3)
print(f"tokenised in {time.time() - t0:.0f}s | tokens: fold1 {sum(len(d['ids']) for d in docs1):,} | fold2 {sum(len(d['ids']) for d in docs2):,} | fold3 {sum(len(d['ids']) for d in docs3):,}")

# offsets must be non-decreasing, otherwise searchsorted alignment is wrong
for d in docs1[:50] + docs3[:50]:
    assert np.all(np.diff(d["starts"]) >= 0) and np.all(np.diff(d["ends"]) >= 0), "token offsets not monotonic"

## 5. Metric, decoding, oracle check

In [ ]:
def strict_scores(gold, pred):
    # gold / pred: {doc_id: [(start, end, label), ...]}; exact start+end+label match
    per_label = {}
    for lab in LABELS:
        tp = 0
        n_pred = 0
        n_gold = 0
        for doc_id in gold:
            g = set(x for x in gold[doc_id] if x[2] == lab)
            p = set(x for x in pred.get(doc_id, []) if x[2] == lab)
            tp += len(g & p)
            n_pred += len(p)
            n_gold += len(g)
        precision = tp / n_pred if n_pred > 0 else 0.0
        recall = tp / n_gold if n_gold > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        per_label[lab] = {"p": precision, "r": recall, "f1": f1, "tp": tp, "pred": n_pred, "gold": n_gold}
    # a label counts in the macro average if it has gold entities or predictions (all 12 do on the real data)
    used = [l for l in LABELS if per_label[l]["gold"] > 0 or per_label[l]["pred"] > 0]
    if not used:
        used = LABELS
    macro = {}
    macro["f1"] = float(np.mean([per_label[l]["f1"] for l in used]))
    macro["p"] = float(np.mean([per_label[l]["p"] for l in used]))
    macro["r"] = float(np.mean([per_label[l]["r"] for l in used]))
    return macro, per_label

# ---- Viterbi with BIO rules ----
def build_transitions():
    NEG = -1e4
    trans = np.zeros((N_TAGS, N_TAGS), dtype=np.float32)     # trans[from, to]
    start = np.zeros(N_TAGS, dtype=np.float32)
    for to in range(N_TAGS):
        if to > 0 and (to - 1) % 2 == 1:                     # to is I-x
            lab = (to - 1) // 2
            allowed = [1 + 2 * lab, 2 + 2 * lab]             # only after B-x or I-x
            for frm in range(N_TAGS):
                if frm not in allowed:
                    trans[frm, to] = NEG
            start[to] = NEG
    return trans, start

TRANS, START = build_transitions()

def viterbi(lp):
    n = lp.shape[0]
    if n == 0:
        return np.zeros(0, dtype=np.int64)
    score = lp[0] + START
    back = np.zeros((n, N_TAGS), dtype=np.int16)
    for t in range(1, n):
        cand = score[:, None] + TRANS
        back[t] = cand.argmax(axis=0)
        score = cand.max(axis=0) + lp[t]
    best = np.zeros(n, dtype=np.int64)
    best[-1] = int(score.argmax())
    for t in range(n - 1, 0, -1):
        best[t - 1] = back[t, best[t]]
    return best

def decode_doc(lp, doc, mode):
    idx = np.where(doc["real"])[0]
    sub = lp[idx]
    if mode == "viterbi":
        tags = viterbi(sub)
    else:
        tags = sub.argmax(axis=1)
    return tags_to_spans(tags, doc["starts"][idx], doc["ends"][idx])

# ---- oracle: gold -> BIO -> spans. Shows the best score this tokenisation can ever reach ----
def oracle_check(docs, name):
    gold = {}
    pred = {}
    for d in docs:
        idx = np.where(d["real"])[0]
        gold[d["id"]] = d["gold"]
        pred[d["id"]] = tags_to_spans(d["tags"][idx], d["starts"][idx], d["ends"][idx])
    macro, per_label = strict_scores(gold, pred)
    print(f"ORACLE {name}: macro-F1 {macro['f1']:.4f} (P {macro['p']:.4f}, R {macro['r']:.4f})")
    bad = [(l, per_label[l]["f1"]) for l in LABELS if per_label[l]["f1"] < 0.99 and per_label[l]["gold"] > 0]
    if bad:
        print("   labels below 0.99:", [(l, round(f, 3)) for l, f in bad])
    return macro["f1"]

o1 = oracle_check(docs1, "fold1")
o2 = oracle_check(docs2, "fold2")
if min(o1, o2) < 0.97:
    print("!! Oracle is low: many gold spans do not start/end on token boundaries. Check END_INCLUSIVE above and the "
          "offset printout below before spending GPU time.")
    for d in docs1[:3]:
        text = texts1[d["id"]]
        for s, e, lab in d["gold"][:3]:
            print(repr(text[max(0, s - 10):s]), "[[", repr(text[s:e]), "]]", lab)

## 6. Windows (long documents are cut into overlapping pieces)

In [ ]:
def make_windows(n):
    # returns list of (start, end, keep_from). Tokens [keep_from, end) are the ones this window "owns";
    # [start, keep_from) is only warm-up context for the causal model.
    # The first window holds MAX_LEN tokens. Every later window gets the document header (first HEADER_TOKENS
    # tokens) put in front of it, so its body is HEADER_TOKENS shorter and the total stays <= MAX_LEN.
    wins = []
    first_end = min(MAX_LEN, n)
    wins.append((0, first_end, 0))
    prev_end = first_end
    body = MAX_LEN - HEADER_TOKENS
    while prev_end < n:
        st = prev_end - OVERLAP
        en = min(st + body, n)
        wins.append((st, en, prev_end))
        prev_end = en
    return wins

def head_len(st):
    # how many header tokens are put in front of a window that starts at token st
    if st > 0 and HEADER_TOKENS > 0:
        return HEADER_TOKENS
    return 0

def train_windows(docs):
    items = []
    for di, d in enumerate(docs):
        n = len(d["ids"])
        if n == 0:
            continue
        for st, en, keep in make_windows(n):
            items.append((di, st, en, keep))
    return items

def has_entity(doc, st, en, keep):
    seg = doc["tags"][keep:en]
    return bool((seg > 0).any())

def make_batch(docs, items, with_labels):
    B = len(items)
    L = max(head_len(st) + en - st for (di, st, en, keep) in items)
    ids = np.full((B, L), PAD_ID, dtype=np.int64)
    labels = np.full((B, L), -100, dtype=np.int64)
    lengths = np.zeros(B, dtype=np.int64)
    soft = None
    smask = None
    if with_labels and any("soft" in docs[di] for (di, st, en, keep) in items):
        soft = np.zeros((B, L, N_TAGS), dtype=np.float16)       # teacher scores
        smask = np.zeros((B, L), dtype=bool)                    # positions where the teacher term is used
    for j, (di, st, en, keep) in enumerate(items):
        d = docs[di]
        h = head_len(st)
        l = en - st
        if h > 0:
            ids[j, :h] = d["ids"][:h]            # document header as context (never trained, never predicted)
        ids[j, h:h + l] = d["ids"][st:en]
        lengths[j] = h + l
        if with_labels:
            t = d["tags"][st:en].copy()
            t[: keep - st] = -100        # warm-up positions are not trained (they are not used at inference)
            labels[j, h:h + l] = t
            if soft is not None and "soft" in d:
                owned = d["real"][st:en].copy()
                owned[: keep - st] = False                       # same positions as the gold labels
                smask[j, h:h + l] = owned
                soft[j, h:h + l] = d["soft"][st:en]
    out = {}
    out["soft"] = torch.from_numpy(soft) if soft is not None else None
    out["smask"] = torch.from_numpy(smask) if smask is not None else None
    out["ids"] = torch.from_numpy(ids)
    out["labels"] = torch.from_numpy(labels)
    out["lengths"] = torch.from_numpy(lengths)
    return out

n_win = len(train_windows(docs1))
print("fold1 windows:", n_win, "| steps per epoch:", math.ceil(n_win / BATCH_SIZE))

# sizes of the FULL run (used for the time estimate printed at the end of a smoke test)
FULL = {}
FULL["steps_per_epoch"] = math.ceil(n_win / BATCH_SIZE)
FULL["steps_per_epoch_all"] = math.ceil((n_win + len(train_windows(docs2))) / BATCH_SIZE)
FULL["val_windows"] = len(train_windows(docs2))
FULL["test_windows"] = len(train_windows(docs3))

if SMOKE_TEST:
    docs1 = docs1[:SMOKE_DOCS]
    docs2 = docs2[:SMOKE_DOCS]
    docs3 = docs3[:SMOKE_DOCS]
    texts3 = {d["id"]: texts3[d["id"]] for d in docs3}
    print(f"SMOKE: using {len(docs1)} / {len(docs2)} / {len(docs3)} docs (train / val / test)")

## 7. Student: Phase-1 GQA backbone (Model 1) + token-classification head

In [ ]:
sys.path.insert(0, "/kaggle/working")
import contextlib
from legal_gqa_model import LegalGQAConfig, LegalGQAForCausalLM
from torch.utils.checkpoint import checkpoint as torch_checkpoint

class NERModel(nn.Module):
    def __init__(self, backbone, n_classes, head="linear", dropout=0.1, n_frozen_blocks=0):
        super().__init__()
        self.backbone = backbone
        d = backbone.config.n_embd
        self.head_type = head
        self.n_frozen_blocks = n_frozen_blocks     # the first N blocks are frozen: run without gradients
        self.drop = nn.Dropout(dropout)
        if head == "bilstm":
            self.lstm = nn.LSTM(d, d // 2, batch_first=True, bidirectional=True)
        self.cls = nn.Linear(d, n_classes)

    def hidden(self, input_ids):
        # same computation as the Phase-1 forward(), but we stop before lm_head
        b = self.backbone
        T = input_ids.shape[1]
        x = b.drop(b.wte(input_ids))
        cos, sin = b._rope_slice(0, T, input_ids.device, x.dtype)
        for i, blk in enumerate(b.blocks):
            if i < self.n_frozen_blocks:
                with torch.no_grad():
                    x = blk(x, cos, sin, None, True, None, False)[0]
            elif self.training and USE_CHECKPOINT:
                def run(h, blk=blk):
                    return blk(h, cos, sin, None, True, None, False)[0]     # causal, right padding needs no mask
                x = torch_checkpoint(run, x, use_reentrant=False)
            else:
                x = blk(x, cos, sin, None, True, None, False)[0]
        return b.ln_f(x)

    def forward(self, input_ids, lengths):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            h = self.hidden(input_ids)
            if self.head_type == "bilstm":
                T = h.shape[1]
                packed = nn.utils.rnn.pack_padded_sequence(h.float(), lengths.cpu(), batch_first=True, enforce_sorted=False)
                out, _ = self.lstm(packed)
                h, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=T)
            h = self.drop(h)
            logits = self.cls(h)
        return logits.float()

LOAD_REPORTED = False

def build_model():
    global LOAD_REPORTED
    backbone, load_info = LegalGQAForCausalLM.from_pretrained(MODEL_PATH, output_loading_info=True)
    if not LOAD_REPORTED:
        missing = [k for k in load_info.get("missing_keys", []) if k != "lm_head.weight"]   # lm_head is tied to wte
        unexpected = list(load_info.get("unexpected_keys", []))
        print("Phase-1 weights loaded | missing keys:", len(missing), "| unexpected keys:", len(unexpected))
        if missing:
            raise RuntimeError("Phase-1 weights were NOT fully loaded, missing: " + str(missing[:5]))
        LOAD_REPORTED = True
    backbone.config.use_cache = False
    backbone.float()                                   # fine-tune in fp32 (fp16 autocast does the fast maths)
    n_blocks = len(backbone.blocks)
    n_frozen = 0
    if FT_MODE == "partial":
        n_frozen = max(0, n_blocks - N_TRAIN_LAYERS)
    for p in backbone.parameters():
        p.requires_grad = True
    if FT_MODE == "partial" or FREEZE_EMBEDDINGS:
        backbone.wte.weight.requires_grad = False       # lm_head is tied to wte, so it stays fixed too
    for blk in backbone.blocks[:n_frozen]:
        for p in blk.parameters():
            p.requires_grad = False
    model = NERModel(backbone, N_TAGS, HEAD, n_frozen_blocks=n_frozen)
    model.to(DEVICE)
    return model

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

_m = build_model()
total_params, trainable_params = count_params(_m)
print(f"STUDENT total {total_params / 1e6:.2f}M | trainable {trainable_params / 1e6:.2f}M ({100 * trainable_params / total_params:.2f}%)")
print(f"FT_MODE={FT_MODE} | blocks {len(_m.backbone.blocks)} (first {_m.n_frozen_blocks} frozen) | embeddings trainable: {_m.backbone.wte.weight.requires_grad} | head: {HEAD}")
del _m
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 7b. Teacher: Model 4 (GQA+MHA+MoE) + LoRMA adapters, and the soft labels

In [ ]:
from model import LegalGQAMoEForCausalLM          # Model 4 backbone (teacher)

# ---------------------------------------------------------------- LoRMA layer (written by hand, PEFT has none)
class LoRMALinear(nn.Module):
    """Frozen linear layer W0 (d_out x d_in) with a low-rank MULTIPLICATIVE update on its output side:
         plus:  W = ((alpha/r) B A + I) W0     B: d_out x r (zeros at start), A: r x d_out (random)
         perm:  W = P(B A) W0                  P rotates row i of B A cyclically by i; starts at P(BA) = I
       so training starts exactly at the Phase-1 weights."""
    def __init__(self, base, r, alpha, dropout, variant):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        d = base.out_features
        self.d, self.r, self.variant = d, r, variant
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.lorma_A = nn.Parameter(torch.zeros(r, d))
        self.lorma_B = nn.Parameter(torch.zeros(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))     # B stays 0 -> (scale*BA + I) = I
        else:
            with torch.no_grad():
                self.lorma_B.normal_(0.0, 0.02)
                self.lorma_B[:, 0] = 1.0                                # first column of B = ones, rest random
                self.lorma_A.zero_()
                self.lorma_A[0, 0] = 1.0                                # A[0,0] = 1, rest 0  ->  BA has ones in column 0 only
            ii = torch.arange(d).unsqueeze(1)
            jj = torch.arange(d).unsqueeze(0)
            self.register_buffer("rot_idx", (jj - ii) % d, persistent=False)   # rotated[i, j] = M[i, (j - i) mod d]

    def forward(self, x):
        if self.variant == "plus":
            z = self.base(x)                                            # W0 x
            return z + self.scale * F.linear(F.linear(self.drop(z), self.lorma_A), self.lorma_B)
        M = self.lorma_B @ self.lorma_A                                 # d x d, rank <= r
        R = M.gather(1, self.rot_idx)                                   # rank-inflated by the row rotation
        w = R.to(self.base.weight.dtype) @ self.base.weight             # P(BA) W0
        b = None if self.base.bias is None else R.to(self.base.bias.dtype) @ self.base.bias
        return F.linear(self.drop(x), w, b)

def add_lorma(root, targets, r, alpha, dropout, variant):
    names = []
    for name, module in list(root.named_modules()):
        for child_name, child in list(module.named_children()):
            if child_name in targets and isinstance(child, nn.Linear):
                setattr(module, child_name, LoRMALinear(child, r, alpha, dropout, variant))
                names.append(name + "." + child_name)
    return names


class TeacherNER(nn.Module):
    # same attribute names as the Model-4 LoRMA notebook's NERModel, so its saved weights load directly
    def __init__(self, backbone, n_classes, head):
        super().__init__()
        self.backbone = backbone
        d = backbone.config.n_embd
        self.head_type = head
        self.drop = nn.Dropout(0.1)
        if head == "bilstm":
            self.lstm = nn.LSTM(d, d // 2, batch_first=True, bidirectional=True)
        self.cls = nn.Linear(d, n_classes)

    def hidden(self, input_ids):
        b = self.backbone
        x = b.drop(b.wte(input_ids))
        cos, sin = b._rope_slice(0, input_ids.shape[1], input_ids.device, x.dtype)
        for blk in b.blocks:
            x = blk(x, cos, sin, None, True, None, False)[0]
        return b.ln_f(x)

    def forward(self, input_ids, lengths):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            h = self.hidden(input_ids)
            if self.head_type == "bilstm":
                T = h.shape[1]
                packed = nn.utils.rnn.pack_padded_sequence(h.float(), lengths.cpu(), batch_first=True, enforce_sorted=False)
                out, _ = self.lstm(packed)
                h, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=T)
            logits = self.cls(h)
        return logits.float()

TEACHER_REPORT = {}
TEACHER_DEFAULTS = {"LORMA_VARIANT": "plus", "LORMA_R": 16, "LORMA_ALPHA": 16, "LORMA_DROPOUT": 0.05,
                    "LORMA_TARGETS": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                    "HEAD": "bilstm", "MAX_LEN": 1024, "OVERLAP": 256, "HEADER_TOKENS": 256, "ADD_EOS_PREFIX": True}

def find_teacher_base():
    if TEACHER_BASE_DIR is not None:
        return TEACHER_BASE_DIR
    found = []
    for cfg_path in find_files("config.json"):
        folder = os.path.dirname(cfg_path)
        try:
            model_type = json.load(open(cfg_path)).get("model_type")
        except Exception:
            continue
        has_weights = os.path.exists(os.path.join(folder, "model.safetensors")) or \
                      os.path.exists(os.path.join(folder, "pytorch_model.bin"))
        if model_type == "legal_gqa_mha_moe" and has_weights and "checkpoint-" not in folder:
            found.append(folder)
    if not found:
        raise FileNotFoundError("Teacher backbone not found. Attach the output of the Model-4 Phase-1 notebook (legal-gqamoe-final).")
    return found[0]

def find_teacher_weights(tag):
    name = "phase%s_lorma_weights.pt" % tag
    if TEACHER_ADAPTER_DIR is not None:
        path = os.path.join(TEACHER_ADAPTER_DIR, name)
        found = [path] if os.path.exists(path) else []
    else:
        found = find_files(name)
    if not found:
        raise FileNotFoundError(name + " not found. Attach the output of the Model-4 LoRMA NER notebook.")
    return found[0]

def load_teacher_settings(weights_path):
    settings = dict(TEACHER_DEFAULTS)
    res_path = os.path.join(os.path.dirname(weights_path), "results.json")
    if os.path.exists(res_path):
        saved = json.load(open(res_path)).get("settings", {})
        for k in settings:
            if k in saved:
                settings[k] = saved[k]
        settings["_from_results_json"] = True
    else:
        print("WARNING: no results.json next to the teacher weights -> assuming the Model-4 notebook defaults", TEACHER_DEFAULTS)
        settings["_from_results_json"] = False
    return settings

TEACHER_BASE = None
def build_teacher(tag):
    global TEACHER_BASE
    if TEACHER_BASE is None:
        TEACHER_BASE = find_teacher_base()
    wpath = find_teacher_weights(tag)
    s = load_teacher_settings(wpath)
    print(f"teacher {tag}: base {TEACHER_BASE}\n           adapters {wpath}\n           LoRMA {s['LORMA_VARIANT']} r={s['LORMA_R']} alpha={s['LORMA_ALPHA']} | head {s['HEAD']} | windows {s['MAX_LEN']}/{s['OVERLAP']}/{s['HEADER_TOKENS']}")
    backbone = LegalGQAMoEForCausalLM.from_pretrained(TEACHER_BASE)
    backbone.config.use_cache = False
    backbone.float()
    for p in backbone.parameters():
        p.requires_grad = False
    model = TeacherNER(backbone, N_TAGS, s["HEAD"])
    add_lorma(backbone, set(s["LORMA_TARGETS"]), s["LORMA_R"], s["LORMA_ALPHA"], s["LORMA_DROPOUT"], s["LORMA_VARIANT"])
    for n, p in model.named_parameters():
        if "lorma_" in n or not n.startswith("backbone."):
            p.requires_grad = True
    state = torch.load(wpath, map_location="cpu")
    trainable_names = {n for n, p in model.named_parameters() if p.requires_grad}
    if set(state.keys()) != trainable_names:
        raise RuntimeError("Teacher weights do not match the rebuilt teacher (different LoRMA/head settings?). "
                           "only in file: %s | only in model: %s" % (sorted(set(state) - trainable_names)[:3], sorted(trainable_names - set(state))[:3]))
    params = dict(model.named_parameters())
    with torch.no_grad():
        for n, v in state.items():
            params[n].copy_(v)
    model.to(DEVICE).eval()
    return model, s

@contextlib.contextmanager
def teacher_windows(s):
    # the teacher reads each document with ITS OWN window settings (restored afterwards)
    names = ["MAX_LEN", "OVERLAP", "HEADER_TOKENS"]
    old = {n: globals()[n] for n in names}
    for n in names:
        globals()[n] = s[n]
    try:
        yield
    finally:
        for n in names:
            globals()[n] = old[n]

@torch.no_grad()
def teacher_logits(model, docs, s, batch_windows=16):
    # raw (pre-softmax) teacher scores for every token, float16
    model.eval()
    out = [np.zeros((len(d["ids"]), N_TAGS), dtype=np.float16) for d in docs]
    with teacher_windows(s):
        items = []
        for di, d in enumerate(docs):
            if len(d["ids"]) == 0:
                continue
            for st, en, keep in make_windows(len(d["ids"])):
                items.append((di, st, en, keep))
        for i in range(0, len(items), batch_windows):
            chunk = items[i:i + batch_windows]
            batch = make_batch(docs, chunk, with_labels=False)
            logits = model(batch["ids"].to(DEVICE), batch["lengths"]).cpu().numpy()
            for j, (di, st, en, keep) in enumerate(chunk):
                h = head_len(st)
                out[di][keep:en] = logits[j, h + keep - st:h + en - st].astype(np.float16)
    return out

def check_same_tokenizer(groups, teacher_base, s):
    t_tok = AutoTokenizer.from_pretrained(teacher_base)
    if t_tok.eos_token_id != tokenizer.eos_token_id:
        raise RuntimeError("teacher and student tokenizers differ (eos id)")
    if s["ADD_EOS_PREFIX"] != ADD_EOS_PREFIX:
        raise RuntimeError("teacher used ADD_EOS_PREFIX=%s, student uses %s: token positions would not line up" % (s["ADD_EOS_PREFIX"], ADD_EOS_PREFIX))
    first = 1 if ADD_EOS_PREFIX else 0
    n_checked = 0
    for docs in groups:
        for d in docs[:25]:
            ids = t_tok(d["text"], add_special_tokens=False)["input_ids"]
            if list(ids) != list(d["ids"][first:]):
                raise RuntimeError("teacher and student tokenise document %s differently: soft labels cannot be aligned" % d["id"])
            n_checked += 1
    print(f"tokenizer check: teacher and student agree on {n_checked} documents")

def attach_soft(tag, train_names, report_names=()):
    """Compute (or load from cache) the teacher scores, attach them to the training documents.
       Returns the list of extra unlabelled documents (empty unless KD_ON_TEST_TEXT)."""
    if KD_ALPHA <= 0:
        print("KD_ALPHA = 0 -> no teacher, gold labels only (baseline)")
        return []
    fold_docs = {"fold1": docs1, "fold2": docs2, "fold3": docs3}
    names = list(train_names) + [n for n in report_names if n not in train_names]
    if KD_ON_TEST_TEXT and "fold3" not in names:
        names.append("fold3")
    cache_path = os.path.join(OUT_DIR, "teacher_logits_%s.npz" % tag)
    cache = {}
    if os.path.exists(cache_path):
        z = np.load(cache_path)
        cache = {k: z[k] for k in z.files}
    wanted = [(n, d) for n in names for d in fold_docs[n]]
    missing = [(n, d) for n, d in wanted if n + "/" + d["id"] not in cache]
    teacher = None
    if missing:
        teacher, s = build_teacher(tag)
        check_same_tokenizer([fold_docs[n] for n in names], TEACHER_BASE, s)
        t0 = time.time()
        for n in names:
            todo = [d for d in fold_docs[n] if n + "/" + d["id"] not in cache]
            if not todo:
                continue
            arrs = teacher_logits(teacher, todo, s)
            for d, a in zip(todo, arrs):
                cache[n + "/" + d["id"]] = a
        tmp = cache_path[:-4] + ".tmp.npz"
        np.savez_compressed(tmp, **cache)
        os.replace(tmp, cache_path)
        print(f"teacher {tag}: scores computed for {len(missing)} documents in {time.time() - t0:.0f}s -> {os.path.basename(cache_path)}")
    else:
        print(f"teacher {tag}: scores loaded from cache ({len(wanted)} documents)")
    for n in train_names:
        for d in fold_docs[n]:
            d["soft"] = cache[n + "/" + d["id"]]
    # how good is the teacher itself on the report folds (fold2 for the fold1-only teacher)?
    for n in report_names:
        docs_r = fold_docs[n]
        preds = {}
        for d in docs_r:
            lp = F.log_softmax(torch.from_numpy(cache[n + "/" + d["id"]].astype(np.float32)), dim=-1).numpy()
            preds[d["id"]] = decode_doc(lp, d, DECODE)
        macro, _ = strict_scores({d["id"]: d["gold"] for d in docs_r}, preds)
        TEACHER_REPORT["report_" + tag] = macro["f1"]
        TEACHER_REPORT[tag] = macro["f1"]
        print(f"TEACHER {tag} on {n}: strict macro-F1 {macro['f1']:.4f}   (the student's fold2 score is compared against this)")
    extra = []
    if KD_ON_TEST_TEXT:
        for d in docs3:
            e = dict(d)
            e["tags"] = np.full(len(d["ids"]), -100, dtype=np.int64)       # no gold labels
            e["soft"] = cache["fold3/" + d["id"]]
            extra.append(e)
        print(f"KD_ON_TEST_TEXT: {len(extra)} unlabelled test documents added to the distillation set")
    if teacher is not None:
        del teacher
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return extra

## 8. Prediction

In [ ]:
@torch.no_grad()
def predict_logprobs(model, docs, batch_windows=16):
    # log-probabilities over the 25 tags for every token of every document
    model.eval()
    logprobs = [np.zeros((len(d["ids"]), N_TAGS), dtype=np.float32) for d in docs]
    items = []
    for di, d in enumerate(docs):
        if len(d["ids"]) == 0:
            continue
        for st, en, keep in make_windows(len(d["ids"])):
            items.append((di, st, en, keep))
    for i in range(0, len(items), batch_windows):
        chunk = items[i:i + batch_windows]
        batch = make_batch(docs, chunk, with_labels=False)
        logits = model(batch["ids"].to(DEVICE), batch["lengths"])
        lps = F.log_softmax(logits, dim=-1).cpu().numpy()
        for j, (di, st, en, keep) in enumerate(chunk):
            h = head_len(st)
            logprobs[di][keep:en] = lps[j, h + keep - st:h + en - st]
    return logprobs

def predict_docs(model, docs, batch_windows=16):
    # returns {doc_id: [(start, end_exclusive, label), ...]}
    logprobs = predict_logprobs(model, docs, batch_windows)
    preds = {}
    for di, d in enumerate(docs):
        preds[d["id"]] = decode_doc(logprobs[di], d, DECODE)
    return preds

def save_logprobs(model, docs, name):
    # per-token log-probs, kept so several models can be ensembled later
    lps = predict_logprobs(model, docs)
    tmp = os.path.join(OUT_DIR, name[:-4] + ".tmp.npz")
    np.savez_compressed(tmp, **{d["id"]: lp.astype(np.float16) for d, lp in zip(docs, lps)})
    os.replace(tmp, os.path.join(OUT_DIR, name))

def evaluate(model, docs):
    pred = predict_docs(model, docs)
    gold = {d["id"]: d["gold"] for d in docs}
    macro, per_label = strict_scores(gold, pred)
    return macro, per_label, pred

# ---- same-name voting inside one document ----
from collections import Counter
VOTE_GROUPS = [["APP", "RESP"], ["A.COUNSEL", "R.COUNSEL"]]     # labels that get confused with each other

def norm_surface(text):
    t = " ".join(text.lower().split())
    return t.strip(" .,;:()[]\"'")

def vote_spans(spans, text):
    # if the same name is predicted several times in a document with a mix of labels from one group,
    # give all of them the majority label (ties are left alone)
    out = list(spans)
    for group in VOTE_GROUPS:
        by_name = {}
        for i in range(len(out)):
            s, e, lab = out[i]
            if lab in group:
                key = norm_surface(text[s:e])
                if key not in by_name:
                    by_name[key] = []
                by_name[key].append(i)
        for key in by_name:
            idxs = by_name[key]
            if len(idxs) < 2:
                continue
            counts = Counter(out[i][2] for i in idxs).most_common()
            if len(counts) > 1 and counts[0][1] == counts[1][1]:
                continue
            winner = counts[0][0]
            for i in idxs:
                out[i] = (out[i][0], out[i][1], winner)
    return out

def postprocess(docs, pred):
    result = {}
    for d in docs:
        result[d["id"]] = vote_spans(pred.get(d["id"], []), d["text"])
    return result

def to_submission(pred):
    out = {}
    for doc_id, spans in pred.items():
        rows = []
        for s, e, lab in sorted(spans):
            end_out = e - 1 if END_INCLUSIVE else e
            rows.append([int(s), int(end_out), lab])
        out[doc_id] = rows
    return out

def check_submission(sub, texts):
    assert set(sub.keys()) == set(texts.keys()), "doc ids differ from the test file"
    n_spans = 0
    for doc_id, rows in sub.items():
        for s, e, lab in rows:
            end_excl = e + 1 if END_INCLUSIVE else e
            assert 0 <= s < end_excl <= len(texts[doc_id]), (doc_id, s, e)
            assert lab in LABEL2ID
            n_spans += 1
    print(f"submission OK: {len(sub)} docs, {n_spans} spans")

def save_submission(sub, name):
    path = os.path.join(OUT_DIR, name)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(sub, f)
    os.replace(tmp, path)
    print("wrote", path)

def show_samples(sub, texts, k=5):
    shown = 0
    for doc_id, rows in sub.items():
        for s, e, lab in rows[:2]:
            end_excl = e + 1 if END_INCLUSIVE else e
            print(f"  {doc_id} {lab:<10} [{s}:{e}] {texts[doc_id][s:end_excl]!r}")
            shown += 1
        if shown >= k:
            break

## 9. Training loop (with checkpoints, session time cap and resume)

In [ ]:
def compute_class_weights(docs):
    counts = np.zeros(N_TAGS, dtype=np.float64)
    for d in docs:
        t = d["tags"]
        t = t[t >= 0]
        counts += np.bincount(t, minlength=N_TAGS)
    if CLASS_WEIGHTS == "none":
        w = np.ones(N_TAGS)
    else:
        total = counts.sum()
        w = np.ones(N_TAGS)
        for c in range(N_TAGS):
            if counts[c] > 0:
                ratio = total / (N_TAGS * counts[c])
                w[c] = ratio ** 0.5 if CLASS_WEIGHTS == "sqrt" else ratio
        w = np.clip(w, 0.05, 20.0)
    return torch.tensor(w, dtype=torch.float32), counts

def trainable_state(model):
    state = {}
    for n, p in model.named_parameters():
        if p.requires_grad:
            state[n] = p.detach().cpu().clone()
    return state

def load_trainable_state(model, state):
    params = dict(model.named_parameters())
    with torch.no_grad():
        for n, v in state.items():
            params[n].copy_(v.to(params[n].device))

def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, default=str)
    os.replace(tmp, path)

def seconds_left():
    return MAX_SESSION_HOURS * 3600 - (time.time() - SESSION_START)

def write_marker(status, note=""):
    marker = {"status": status, "note": note, "saved_at": time.time(),
              "saved_at_text": time.strftime("%Y-%m-%d %H:%M:%S"),
              "fingerprint": settings_fingerprint(), "experiment": EXPERIMENT_NAME}
    atomic_json(marker, os.path.join(OUT_DIR, "resume_marker.json"))

def train_model(train_docs, n_epochs, val_docs=None, tag="A"):
    set_seed(SEED)
    ckpt_path = os.path.join(OUT_DIR, "ckpt_" + tag + ".pt")
    fingerprint = settings_fingerprint()
    fingerprint["tag"] = tag
    fingerprint["n_epochs"] = n_epochs
    fingerprint["n_train_docs"] = len(train_docs)

    model = build_model()
    class_w, counts = compute_class_weights(train_docs)
    print("class weights:", {TAG_NAMES[i]: round(float(class_w[i]), 2) for i in range(N_TAGS) if i < 5})
    loss_fn = nn.CrossEntropyLoss(weight=class_w.to(DEVICE), ignore_index=-100)

    body_decay, body_nodecay, head_params = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("backbone."):
            (body_decay if p.ndim >= 2 else body_nodecay).append(p)
        else:
            head_params.append(p)
    groups = [{"params": head_params, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY}]
    if body_decay:
        groups.append({"params": body_decay, "lr": LR_BODY, "weight_decay": WEIGHT_DECAY})
    if body_nodecay:
        groups.append({"params": body_nodecay, "lr": LR_BODY, "weight_decay": 0.0})
    optimizer = torch.optim.AdamW(groups)

    all_items = train_windows(train_docs)
    if SMOKE_TEST:
        smoke_rng = random.Random(SEED)
        smoke_rng.shuffle(all_items)
        all_items = all_items[:SMOKE_MAX_WINDOWS]
    steps_per_epoch = math.ceil(len(all_items) / BATCH_SIZE)      # upper bound when NEG_KEEP_PROB < 1
    total_steps = steps_per_epoch * n_epochs
    warmup_steps = max(1, int(WARMUP_FRAC * total_steps))

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    history = []
    best_f1 = -1.0
    best_epoch = 0
    best_state = None
    bad_epochs = 0
    step = 0
    start_epoch = 1
    seconds_before = 0.0
    finished = False
    stopped_for_time = False

    # ---------- resume from a checkpoint of an earlier session ----------
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        if ck["fingerprint"] != fingerprint:
            raise RuntimeError("Checkpoint " + ckpt_path + " was made with different settings.\n saved:   " +
                               str(ck["fingerprint"]) + "\n current: " + str(fingerprint) +
                               "\nRevert the settings, or delete the checkpoint to start this phase again.")
        load_trainable_state(model, ck["model"])
        if ck["optimizer"] is not None:
            optimizer.load_state_dict(ck["optimizer"])
            scheduler.load_state_dict(ck["scheduler"])
            scaler.load_state_dict(ck["scaler"])
        history = ck["history"]
        best_f1 = ck["best_f1"]
        best_epoch = ck["best_epoch"]
        best_state = ck["best_state"]
        bad_epochs = ck["bad_epochs"]
        step = ck["step"]
        start_epoch = ck["epoch"] + 1
        seconds_before = ck["seconds"]
        finished = ck["finished"]
        random.setstate(ck["rng"]["python"])
        np.random.set_state(ck["rng"]["numpy"])
        torch.set_rng_state(ck["rng"]["torch"])
        if DEVICE.type == "cuda" and ck["rng"]["cuda"] is not None:
            torch.cuda.set_rng_state_all(ck["rng"]["cuda"])
        print(f"[{tag}] RESUMED from checkpoint: {ck['epoch']} of {n_epochs} epochs already done" +
              (" (phase already finished)" if finished else ""))

    t_start = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    for epoch in range(start_epoch, n_epochs + 1):
        if finished:
            break
        model.train()
        t_epoch = time.time()
        items = []
        for it in all_items:
            di, st, en, keep = it
            if NEG_KEEP_PROB < 1.0 and not has_entity(train_docs[di], st, en, keep):
                if random.random() > NEG_KEEP_PROB:
                    continue
            items.append(it)
        random.shuffle(items)
        running = 0.0
        n_batches = 0
        for i in range(0, len(items), BATCH_SIZE):
            chunk = items[i:i + BATCH_SIZE]
            optimizer.zero_grad()
            loss_value = 0.0
            for j in range(0, len(chunk), MICRO_BATCH):
                sub_chunk = chunk[j:j + MICRO_BATCH]
                frac = len(sub_chunk) / len(chunk)
                batch = make_batch(train_docs, sub_chunk, with_labels=True)
                logits = model(batch["ids"].to(DEVICE), batch["lengths"])
                labels_t = batch["labels"].to(DEVICE)
                valid = labels_t != -100
                if bool(valid.any()):
                    hard = loss_fn(logits[valid], labels_t[valid])
                else:
                    hard = logits.sum() * 0.0                     # unlabelled test windows: no gold term
                loss = hard
                if KD_ALPHA > 0 and batch["smask"] is not None and bool(batch["smask"].any()):
                    sm = batch["smask"].to(DEVICE)
                    s_scores = logits[sm] / KD_TEMP
                    t_scores = batch["soft"].to(DEVICE)[sm].float() / KD_TEMP
                    kd = F.kl_div(F.log_softmax(s_scores, dim=-1), F.softmax(t_scores, dim=-1), reduction="batchmean") * (KD_TEMP ** 2)
                    loss = (1.0 - KD_ALPHA) * hard + KD_ALPHA * kd
                scaler.scale(loss * frac).backward()
                loss_value += float(loss.item()) * frac
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            step += 1
            running += loss_value
            n_batches += 1
            if n_batches % 50 == 0:
                print(f"  [{tag}] epoch {epoch} batch {n_batches}/{math.ceil(len(items) / BATCH_SIZE)} loss {running / n_batches:.4f}")
        row = {"epoch": epoch, "train_loss": running / max(1, n_batches),
               "seconds": seconds_before + time.time() - t_start}
        row["train_seconds"] = time.time() - t_epoch
        row["train_batches"] = n_batches

        if val_docs is not None:
            t_eval = time.time()
            macro, per_label, _ = evaluate(model, val_docs)
            row["eval_seconds"] = time.time() - t_eval
            row["val_windows"] = len(train_windows(val_docs))
            row["val_f1"] = macro["f1"]
            row["val_p"] = macro["p"]
            row["val_r"] = macro["r"]
            print(f"[{tag}] epoch {epoch}: loss {row['train_loss']:.4f} | val strict macro-F1 {macro['f1']:.4f} (P {macro['p']:.4f} R {macro['r']:.4f}) | {row['seconds'] / 60:.1f} min")
            if macro["f1"] > best_f1:
                best_f1 = macro["f1"]
                best_epoch = epoch
                best_state = trainable_state(model)
                bad_epochs = 0
            else:
                bad_epochs += 1
        else:
            print(f"[{tag}] epoch {epoch}: loss {row['train_loss']:.4f} | {row['seconds'] / 60:.1f} min")
        history.append(row)

        patience_stop = val_docs is not None and bad_epochs >= PATIENCE
        if patience_stop or epoch == n_epochs:
            finished = True

        # ---------- checkpoint after every epoch (written atomically) ----------
        ck = {}
        ck["fingerprint"] = fingerprint
        ck["model"] = trainable_state(model)
        ck["optimizer"] = optimizer.state_dict()
        ck["scheduler"] = scheduler.state_dict()
        ck["scaler"] = scaler.state_dict()
        ck["history"] = history
        ck["best_f1"] = best_f1
        ck["best_epoch"] = best_epoch
        ck["best_state"] = best_state
        ck["bad_epochs"] = bad_epochs
        ck["step"] = step
        ck["epoch"] = epoch
        ck["seconds"] = seconds_before + time.time() - t_start
        ck["finished"] = finished
        ck["rng"] = {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
                     "cuda": torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else None}
        if finished:
            # a finished phase does not need optimizer state: keep the checkpoint small
            ck["optimizer"] = None
            ck["scheduler"] = None
            ck["scaler"] = None
            if best_state is not None:
                ck["model"] = best_state
        atomic_save(ck, ckpt_path)
        write_marker("in_progress", f"phase {tag}: epoch {epoch}/{n_epochs} done")

        if finished:
            if patience_stop:
                print(f"[{tag}] no improvement for {PATIENCE} epochs, stopping")
            break

        # ---------- session time cap ----------
        epoch_total = time.time() - t_epoch
        needed = 1.3 * epoch_total + RESERVE_MIN * 60
        if seconds_left() < needed:
            stopped_for_time = True
            print(f"[{tag}] TIME CAP: {seconds_left() / 60:.0f} min left, next epoch + final steps need ~{needed / 60:.0f} min. Stopping after epoch {epoch}.")
            break

    info = {}
    info["history"] = history
    info["best_epoch"] = best_epoch
    info["best_val_f1"] = best_f1
    info["train_seconds"] = seconds_before + time.time() - t_start
    info["peak_gpu_gb"] = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0
    info["finished"] = finished
    info["stopped_for_time"] = stopped_for_time
    if best_state is not None:
        load_trainable_state(model, best_state)
    return model, info

def save_model_fp16(model, name):
    state = {k: v.detach().half().cpu() for k, v in model.state_dict().items() if not k.endswith("lm_head.weight")}
    atomic_save(state, os.path.join(OUT_DIR, name))

def drop_ckpt(tag):
    path = os.path.join(OUT_DIR, "ckpt_" + tag + ".pt")
    if os.path.exists(path):
        os.remove(path)
        print("removed", os.path.basename(path), "(phase result is saved, the large checkpoint is no longer needed)")

def pause_message(phase):
    print("\n" + "=" * 70)
    print("SESSION TIME CAP REACHED during phase " + phase + ". Everything needed to continue is saved in:")
    print("   " + OUT_DIR)
    print("TO CONTINUE: when this version has finished, open the notebook again, Add Input -> this notebook's")
    print("output, keep the SAME settings, and Save & Run All. It resumes where it stopped.")
    print("(a usable submission.json from the best epoch so far has already been written)")
    print("=" * 70)

## 10. Phase A – train on fold1, validate on fold2

In [ ]:
PHASE_A_PATH = os.path.join(OUT_DIR, "phaseA_result.json")
model_a = None
info_a = None
if os.path.exists(PHASE_A_PATH):
    print("Phase A was already finished in an earlier session -> skipped (its result is loaded from file).")
else:
    extra_a = attach_soft("A", ["fold1"], report_names=["fold2"])
    model_a, info_a = train_model(docs1 + extra_a, EPOCHS, val_docs=docs2, tag="A")
    print("best epoch:", info_a["best_epoch"], "| best val strict macro-F1:", round(info_a["best_val_f1"], 4))
    print(f"train time {info_a['train_seconds'] / 60:.1f} min | peak GPU memory {info_a['peak_gpu_gb']:.2f} GB")
    save_model_fp16(model_a, "phaseA_model_fp16.pt")
    save_logprobs(model_a, docs2, "logprobs_fold2_phaseA.npz")
    save_logprobs(model_a, docs3, "logprobs_fold3_phaseA.npz")
    if info_a["stopped_for_time"]:
        OUT_OF_TIME = True
        partial = to_submission(predict_docs(model_a, docs3))        # best epoch so far
        check_submission(partial, texts3)
        save_submission(partial, "submission_partial_phaseA.json")
        save_submission(partial, "submission.json")
        write_marker("paused", "phase A stopped for time")
        pause_message("A")

### Per-label scores and error analysis on fold2 (for the report)

In [ ]:
if not OUT_OF_TIME:
    if model_a is not None:
        macro_a, per_label_a, pred_a = evaluate(model_a, docs2)
        print(f"FOLD2 strict macro-F1 {macro_a['f1']:.4f} | macro-P {macro_a['p']:.4f} | macro-R {macro_a['r']:.4f}\n")
        print(f"{'label':<10} {'P':>6} {'R':>6} {'F1':>6} {'gold':>6} {'pred':>6}")
        for lab in LABELS:
            r = per_label_a[lab]
            print(f"{lab:<10} {r['p']:6.3f} {r['r']:6.3f} {r['f1']:6.3f} {r['gold']:6d} {r['pred']:6d}")

        def overlaps(a, b):
            return a[0] < b[1] and b[0] < a[1]

        def error_analysis(docs, pred):
            c = {"correct": 0, "boundary_error": 0, "label_confusion": 0, "missed": 0,
                 "spurious": 0, "boundary_and_label": 0}
            confusion = {}
            for d in docs:
                g_list = d["gold"]
                p_list = pred.get(d["id"], [])
                g_set = set(g_list)
                p_set = set(p_list)
                for g in g_list:
                    if g in p_set:
                        c["correct"] += 1
                        continue
                    same_label = [p for p in p_list if overlaps(g, p) and p[2] == g[2]]
                    diff_label_same_span = [p for p in p_list if p[0] == g[0] and p[1] == g[1] and p[2] != g[2]]
                    diff_label = [p for p in p_list if overlaps(g, p) and p[2] != g[2]]
                    if same_label:
                        c["boundary_error"] += 1
                    elif diff_label_same_span:
                        c["label_confusion"] += 1
                        key = (g[2], diff_label_same_span[0][2])
                        confusion[key] = confusion.get(key, 0) + 1
                    elif diff_label:
                        c["boundary_and_label"] += 1
                        key = (g[2], diff_label[0][2])
                        confusion[key] = confusion.get(key, 0) + 1
                    else:
                        c["missed"] += 1
                for p in p_list:
                    if p in g_set:
                        continue
                    if not any(overlaps(p, g) for g in g_list):
                        c["spurious"] += 1
            return c, confusion

        errors_a, confusion_a = error_analysis(docs2, pred_a)
        print("\nerror breakdown (gold entities -> outcome; spurious = predictions touching no gold entity):")
        for k, v in errors_a.items():
            print(f"  {k:<20} {v}")
        print("\ntop label confusions (gold -> predicted):")
        for (g, p), n in sorted(confusion_a.items(), key=lambda x: -x[1])[:10]:
            print(f"  {g:<10} -> {p:<10} {n}")

        # ---- same-name voting: does it help on fold2? ----
        pred_a_vote = postprocess(docs2, pred_a)
        gold2 = {d["id"]: d["gold"] for d in docs2}
        macro_a_vote, per_label_a_vote = strict_scores(gold2, pred_a_vote)
        print(f"\nSAME-NAME VOTING on fold2: macro-F1 {macro_a['f1']:.4f} -> {macro_a_vote['f1']:.4f}")
        for lab in ["APP", "RESP", "A.COUNSEL", "R.COUNSEL"]:
            print(f"  {lab:<10} F1 {per_label_a[lab]['f1']:.3f} -> {per_label_a_vote[lab]['f1']:.3f}")
        if VOTE == "auto":
            USE_VOTE = macro_a_vote["f1"] > macro_a["f1"]
        else:
            USE_VOTE = bool(VOTE)
        print("voting will be", "USED" if USE_VOTE else "NOT used", "for the test predictions")

## 11. Test predictions from the Phase A model (also the fallback submission)

In [ ]:
if not OUT_OF_TIME:
    if model_a is not None:
        t0 = time.time()
        test_pred_a = predict_docs(model_a, docs3)
        if USE_VOTE:
            test_pred_a = postprocess(docs3, test_pred_a)
        infer_seconds = time.time() - t0
        sub_a = to_submission(test_pred_a)
        check_submission(sub_a, texts3)
        show_samples(sub_a, texts3)
        save_submission(sub_a, "submission_phaseA_fold1only.json")
        save_submission(sub_a, "submission.json")          # fallback until Phase B finishes and replaces it
        print(f"test inference: {infer_seconds:.0f}s")

        phaseA = {"best_epoch": info_a["best_epoch"], "val_macro_f1": macro_a["f1"], "val_macro_p": macro_a["p"],
                  "val_macro_r": macro_a["r"], "per_label": per_label_a, "errors": errors_a,
                  "use_vote": USE_VOTE, "teacher_fold2_f1": TEACHER_REPORT.get("A"), "val_macro_f1_voted": macro_a_vote["f1"], "per_label_voted": per_label_a_vote,
                  "history": info_a["history"], "train_seconds": info_a["train_seconds"],
                  "peak_gpu_gb": info_a["peak_gpu_gb"], "test_inference_seconds": infer_seconds}
        atomic_json(phaseA, PHASE_A_PATH)                  # marks Phase A as finished for any later session
        drop_ckpt("A")
        write_marker("phaseA_done", "phase A finished")

## 12. Phase B – refit on fold1 + fold2 with the best epoch count, final submission

In [ ]:
if not OUT_OF_TIME:
    phaseA = json.load(open(PHASE_A_PATH))
    USE_VOTE = phaseA.get("use_vote", False)
    PHASE_B_PATH = os.path.join(OUT_DIR, "phaseB_result.json")
    phaseB = None
    if REFIT_ON_ALL:
        if os.path.exists(PHASE_B_PATH):
            print("Phase B was already finished in an earlier session -> skipped.")
            phaseB = json.load(open(PHASE_B_PATH))
        else:
            model_a = None
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            n_epochs_b = max(1, phaseA["best_epoch"])
            extra_b = attach_soft("B", ["fold1", "fold2"])
            model_b, info_b = train_model(docs1 + docs2 + extra_b, n_epochs_b, val_docs=None, tag="B")
            if info_b["stopped_for_time"]:
                OUT_OF_TIME = True
                write_marker("paused", "phase B stopped for time")
                pause_message("B")
            else:
                save_model_fp16(model_b, "phaseB_model_fp16.pt")
                save_logprobs(model_b, docs3, "logprobs_fold3_phaseB.npz")
                test_pred_b = predict_docs(model_b, docs3)
                if USE_VOTE:
                    test_pred_b = postprocess(docs3, test_pred_b)
                final_sub = to_submission(test_pred_b)
                check_submission(final_sub, texts3)
                show_samples(final_sub, texts3)
                save_submission(final_sub, "submission.json")
                phaseB = {"epochs": n_epochs_b, "train_seconds": info_b["train_seconds"], "peak_gpu_gb": info_b["peak_gpu_gb"]}
                atomic_json(phaseB, PHASE_B_PATH)
                drop_ckpt("B")
    else:
        print("REFIT_ON_ALL is False: submission.json is the Phase A (fold1 only) model")

## 13. Save results

In [ ]:
if not OUT_OF_TIME:
    results = {}
    results["experiment"] = EXPERIMENT_NAME
    results["settings"] = settings_fingerprint()
    results["end_inclusive"] = END_INCLUSIVE
    results["oracle_f1"] = {"fold1": o1, "fold2": o2}
    results["params_total"] = total_params
    results["params_trainable"] = trainable_params
    results["phaseA"] = phaseA
    results["phaseB"] = phaseB
    results["kd"] = {"alpha": KD_ALPHA, "temperature": KD_TEMP, "on_test_text": KD_ON_TEST_TEXT, "teacher_fold2_f1": TEACHER_REPORT.get("A")}
    atomic_json(results, os.path.join(OUT_DIR, "results.json"))

    import zipfile
    with zipfile.ZipFile(os.path.join(OUT_DIR, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(OUT_DIR, "submission.json"), arcname="prediction.json")     # Codabench wants prediction.json inside submission.zip
    write_marker("complete", "all phases finished")
    print("\nDONE. Files in", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))

    if SMOKE_TEST:
        first = phaseA["history"][0]
        sec_per_step = first["train_seconds"] / max(1, first["train_batches"])
        est_train_epoch = sec_per_step * FULL["steps_per_epoch"]
        est_eval_epoch = first.get("eval_seconds", 0.0) / max(1, first.get("val_windows", 1)) * FULL["val_windows"]
        est_phase_a = (est_train_epoch + est_eval_epoch) * FULL_EPOCHS
        est_phase_b = sec_per_step * FULL["steps_per_epoch_all"] * FULL_EPOCHS      # upper bound (uses all FULL_EPOCHS)
        print("\n" + "=" * 60)
        print("SMOKE TEST PASSED: training, validation, error analysis, test prediction, refit and file writing all ran.")
        print(f"  speed: {sec_per_step:.2f} s/step | peak GPU memory {phaseA['peak_gpu_gb']:.2f} GB (batch {BATCH_SIZE} x {MAX_LEN} tokens)")
        print(f"  full run estimate with EPOCHS={FULL_EPOCHS}:")
        print(f"    phase A: {est_phase_a / 60:.0f} min ({est_train_epoch / 60:.1f} min train + {est_eval_epoch / 60:.1f} min eval per epoch)")
        print(f"    phase B: up to {est_phase_b / 60:.0f} min")
        print(f"    total  : up to {(est_phase_a + est_phase_b) / 3600:.1f} h  (session cap is {MAX_SESSION_HOURS} h)")
        print("  NEXT: set SMOKE_TEST = False and run the whole notebook again.")
        print("=" * 60)

**Upload `submission.zip` (it contains `prediction.json`) to Codabench.** If the first upload scores near 0, the usual cause is the end-offset convention: flip `END_INCLUSIVE` in section 3 and re-run.
`results.json` has everything for the report (per-label scores, error breakdown, params, memory, time).

**If the session time cap stopped the run** ("SESSION TIME CAP REACHED" in the log): open the notebook again, *Add Input* → this notebook's own output, keep the same settings, *Save & Run All*. It continues from the last finished epoch.